In [ ]:
import pandas as pd
import pandas_gbq
from google.cloud import bigquery
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

print("SIA Wevengers 분석 환경 준비 완료")

In [ ]:
from spacetrack import SpaceTrackClient

st = SpaceTrackClient(
    identity="yulihyeon06@gmail.com",
    password="Rosa900108liu860606"
)

# tle_latest → gp 로 변경
try:
    result = list(st.gp(
        norad_cat_id=25544,   # ISS
        orderby="epoch desc",
        limit=1,
        format="tle"
    ))
    print("성공!")
    print(result[0] if result else "데이터 없음")
except Exception as e:
    print(f"실패: {repr(e)}")

In [ ]:
"""
Step 1 — 2022-08-04 활성 지구관측 위성 리스트 수집 및 저장
잔해물(DEBRIS)·로켓바디(ROCKET BODY) 제외, PAYLOAD만
고도 300~1200km + 경사각 40° 이상 = 지구관측 후보

사전 조건: st = SpaceTrackClient(...) 노트북에서 미리 생성
"""

import json
import time
import pandas as pd

# ════════════════════════════════════════════════════════════════
# NORAD ID 구간별 PAYLOAD TLE 수집
# ════════════════════════════════════════════════════════════════
NORAD_RANGES = [
    (1,     10000),
    (10001, 20000),
    (20001, 30000),
    (30001, 40000),
    (40001, 54000),   # 2022년 당시 최신 위성 상한
]

print("=" * 55)
print("Step 1 — 2022-08-04 활성 지구관측 위성 수집")
print("조건: PAYLOAD + 고도 300~1200km + 경사각 40°+")
print("=" * 55)

all_tle = []
for n_from, n_to in NORAD_RANGES:
    try:
        raw     = list(st.gp_history(
            epoch=">2022-08-03,<2022-08-05",
            norad_cat_id=f"{n_from}--{n_to}",
            object_type="PAYLOAD",
            orderby="norad_cat_id asc",
            limit=5000,
            format="json"
        ))
        records = json.loads("".join(raw))

        for rec in records:
            l1 = rec.get("TLE_LINE1", "")
            l2 = rec.get("TLE_LINE2", "")
            if not (l1.startswith("1 ") and l2.startswith("2 ")):
                continue

            incl    = float(rec.get("INCLINATION",    0))
            alt_km  = float(rec.get("SEMIMAJOR_AXIS", 6371)) - 6371

            # 지구관측 필터
            if not (300 <= alt_km <= 1200 and incl >= 40):
                continue

            all_tle.append({
                "name":        rec.get("OBJECT_NAME", "").strip(),
                "norad_id":    int(rec.get("NORAD_CAT_ID", 0)),
                "country":     rec.get("COUNTRY_CODE", ""),
                "epoch":       rec.get("EPOCH", ""),
                "inclination": round(incl, 2),
                "altitude_km": round(alt_km, 1),
                "period_min":  round(float(rec.get("PERIOD", 0)), 2),
                "rcs_size":    rec.get("RCS_SIZE", ""),
                "line1":       l1,
                "line2":       l2,
            })

        print(f"  {n_from:>6}~{n_to:<6}: {len(records):>5}건 수집  "
              f"지구관측 후보 누적 {len(all_tle):,}기")
        time.sleep(1)   # rate limit 방지

    except Exception as e:
        print(f"  {n_from}~{n_to} 실패: {repr(e)}")

print(f"\n수집 완료: 총 {len(all_tle):,}기")

# ── DataFrame 저장 ───────────────────────────────────────────────
eo_sat_df = pd.DataFrame(all_tle)

print(f"\n[기초 통계]")
print(f"  위성 수:      {len(eo_sat_df):,}기")
print(f"  고도 범위:    {eo_sat_df['altitude_km'].min():.1f} ~ "
      f"{eo_sat_df['altitude_km'].max():.1f} km")
print(f"  경사각 범위:  {eo_sat_df['inclination'].min():.1f} ~ "
      f"{eo_sat_df['inclination'].max():.1f}°")

print(f"\n[국가별 위성 수 TOP 15]")
print(eo_sat_df["country"].value_counts().head(15).to_string())

print(f"\n[샘플 10개]")
print(eo_sat_df[["name","norad_id","country",
                  "altitude_km","inclination"]].head(10).to_string(index=False))

# CSV 저장 (TLE 포함)
eo_sat_df.to_csv("eo_satellites_20220804.csv",
                 index=False, encoding="utf-8-sig")
print(f"\n→ eo_satellites_20220804.csv 저장 완료 ({len(eo_sat_df):,}기)")

# JSON 저장 (TLE 포함 — Step 2에서 로드용)
import json as _json
eo_sat_df.to_json("eo_satellites_20220804.json",
                  orient="records", force_ascii=False, indent=2)
print(f"→ eo_satellites_20220804.json 저장 완료")